model training

In [1]:
import pandas as pd
import os
import numpy as np

cancerous = ['BCC','MEL','SCC']
not_cancerous = ['ACK', 'NEV', 'SEK']

In [2]:
df = pd.read_csv('../data/metadata.csv')
df.head()



,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,False,False,False,False,False,False,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,True,True,False,True,True,True,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,True,False,False,False,False,False,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,True,True,False,False,True,True,PAT_684_1302_588.png,True


In [3]:
mask_dir = "../data/masks"

necessary_df = df[
    df['img_id'].apply(lambda x: os.path.exists(f"{mask_dir}/{x.removesuffix(".png")}_mask.png"))][['img_id', 'diagnostic']].copy()
necessary_df['cancerous'] = necessary_df['diagnostic'].isin(cancerous)

df_sample = necessary_df.sample(n=200, random_state=42)
df_sample.head()


,img_id,diagnostic,cancerous
2017,PAT_537_1014_452.png,SCC,True
382,PAT_770_1451_136.png,SCC,True
1775,PAT_1247_852_178.png,ACK,False
310,PAT_270_1382_561.png,SEK,False
200,PAT_1298_1051_831.png,NEV,False


In [4]:
print("CWD:", os.getcwd())
print("Mask dir exists:", os.path.exists("../data/masks"))
print(os.listdir(mask_dir)[:10])

CWD: d:\aa-DS-project-tigers\2026-PDS-Tigers\src
Mask dir exists: True
['.DS_Store', '.gitkeep', 'PAT_1000_31_620_mask.png', 'PAT_1006_53_385_mask.png', 'PAT_1006_53_716_mask.png', 'PAT_1008_59_297_mask.png', 'PAT_100_393_898_mask.png', 'PAT_1013_82_876_mask.png', 'PAT_1014_86_861_mask.png', 'PAT_1017_97_577_mask.png']


In [5]:
from asymmetry import extract_asymmetry  
from border import border
from color import color_features_extraction

In [6]:
import numpy as np

def safe_color_features_extraction(img_id):
    try:
        return color_features_extraction(img_id)
    except ValueError:  # catches the "more dimensions than allowed" error
        return np.nan

# Apply the safe version
df_sample['color'] = df_sample['img_id'].apply(safe_color_features_extraction)


In [7]:
df_sample.head()
nan_count = df_sample['color'].isna().sum()
nan_count


np.int64(7)

In [8]:
df_sample['assymetry'] = df_sample['img_id'].apply(extract_asymmetry)


In [9]:
df_sample['border'] = df_sample['img_id'].apply(border)

d:\aa-DS-project-tigers\2026-PDS-Tigers\src\border.py:35: FutureWarning: `binary_erosion` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.erosion` instead. Note the pixel shift by 1 for even-sized footprints (see docstring notes).
  mask_eroded = morphology.binary_erosion(mask, struct_el)


In [10]:
df_sample.head()

,img_id,diagnostic,cancerous,color,assymetry,border
2017,PAT_537_1014_452.png,SCC,True,"[0.355473, 0.012221703, 0.52124655, 0.42906058]",0.087037,"[0.0013159564272671378, 0.47775545939487923]"
382,PAT_770_1451_136.png,SCC,True,"[0.8211169, 0.009375242, 0.030223852, 0.042998...",0.808419,"[0.008084333237942172, 0.2299716144924801]"
1775,PAT_1247_852_178.png,ACK,False,"[0.7957027, 0.00076159555, 0.002436745, 0.0286...",0.176000,"[0.0030103754396765187, 0.7100731898356365]"
310,PAT_270_1382_561.png,SEK,False,"[0.52810955, 0.008360033, 0.35172004, 0.40228927]",0.360835,"[0.016464739695619533, 0.47371669103480923]"
200,PAT_1298_1051_831.png,NEV,False,"[0.035033777, 0.04308305, 0.037293155, 0.01404...",0.120743,"[0.007959531648638956, 0.399021737566716]"


In [19]:
df_sample = df_sample.dropna()

In [20]:
color_df = pd.DataFrame(df_sample['color'].tolist(), columns=['h_sin', 'rgb_var_mag', 'hsv_var_mag', 'circular_max_min_'])
color_df

,h_sin,rgb_var_mag,hsv_var_mag,circular_max_min_
0,0.355473,0.012222,0.521247,0.429061
1,0.821117,0.009375,0.030224,0.042998
2,0.795703,0.000762,0.002437,0.028658
3,0.528110,0.008360,0.351720,0.402289
4,0.035034,0.043083,0.037293,0.014049
...,...,...,...,...
188,0.187459,0.010663,0.026471,0.110371
189,0.596256,0.039368,0.183450,0.313483
190,0.492770,0.004676,0.010201,0.042932
191,0.553087,0.028364,0.047663,0.152721


In [21]:
border_df = pd.DataFrame(df_sample['border'].tolist(), columns=['compactness', 'convexity'])
border_df

,compactness,convexity
0,0.001316,0.477755
1,0.008084,0.229972
2,0.003010,0.710073
3,0.016465,0.473717
4,0.007960,0.399022
...,...,...
188,0.008582,0.291835
189,0.000697,0.346537
190,0.002393,0.468548
191,0.011969,0.581543


In [25]:
color_df.shape

(193, 4)

In [31]:



from sklearn.model_selection import train_test_split

x = pd.concat([color_df.reset_index(drop=True), df_sample[['assymetry']].reset_index(drop=True), border_df.reset_index(drop=True)], axis=1)
y = df_sample['cancerous']

dev_x, test_x, dev_y, test_y = train_test_split(
        x, y, stratify=y, random_state=0)

train_x, val_x, train_y, val_y = train_test_split(
        dev_x, dev_y, stratify=dev_y)







#### standardisation

### Plotting based on features